Flujo Directo
Análisis realizado en EDA

# Obtención datos

In [ ]:
import sys, os
from supabase import create_client, Client
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
# import plotly.express as px
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath('..'))

from etl.extract import extract_supabase

load_dotenv()
url= os.environ.get("SUPABASE_URL")
key= os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

endpoints= [ "products","customers","orders","orders_products"]
esquema='clean'
df_clean= {}
for endpoint in tqdm(endpoints):
    df_clean[f'{endpoint}_clean']= extract_supabase(supabase=supabase,endpoint=endpoint, esquema=esquema)

customers= df_clean['customers_clean']
orders= df_clean['orders_clean']


Obtencion de RFM

In [ ]:
ventas= orders[orders['estado_orden']=='Pagada']

clientes_ventas= pd.merge(customers, ventas, on='id_cliente', how='left')
clientes_ventas=clientes_ventas[['id_cliente', 'acepta_marketing','id_orden',
       'fecha_creacion', 'precio_total', 'estado_cumplimiento',
       'estado_orden', 'estado_envio', 'estado_cliente']]
# precio a 0 para los que nunca han comprado
clientes_ventas['precio_total']= clientes_ventas['precio_total'].apply(lambda x: 0 if pd.isna(x) else x)

compras_cliente= clientes_ventas.groupby('id_cliente')['id_orden'].count().reset_index().sort_values('id_orden', ascending=False)  
# Frecuency
compras_cliente.columns=['id_cliente','nro_compras'] #renombra 
